# CHART/OPS — Qwen3-VL ChartQA Demo（Colab A100）

這是作品集展示用的 GPU demo：從 HF Hub 載入本專案的 AWQ W4A16 模型，以 vLLM 推論並啟動 Gradio share link。

1. Colab → 執行階段 → 變更執行階段類型 → **A100 GPU**
2. 左側 Secrets 開啟 `HF_TOKEN` 的筆記本存取權
3. 全部執行，不需要修改 code；最後一格會產生可分享的暫時網址
4. 最後一格會持續顯示執行中，這代表 Gradio 服務正常；不要按停止，直到測試完成

> 前置：`<HF_USER>/qwen3vl-8b-chartqa-awq` 已完成正式量化；正式上架時再將相同 UI 搬到 HF Space。


In [ ]:
# 1. GPU 與磁碟檢查（vLLM 安裝前不要 import torch）
import shutil, subprocess, sys

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True, check=True,
).stdout.strip()
gpu_memory_total_mb = int(subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True,
).stdout.strip())
disk_free_gb = shutil.disk_usage("/content").free / 1024**3
print("GPU:", gpu)
print(f"Disk free: {disk_free_gb:.1f} GB")
assert "A100" in gpu and 39000 <= gpu_memory_total_mb <= 42000, "請選擇 A100 40GB runtime 後重新全部執行。"
assert sys.version_info[:2] == (3, 12), f"需要 Colab Python 3.12，目前為 {sys.version.split()[0]}"
assert disk_free_gb >= 25, "可用磁碟需至少 25GB。"

# 在安裝大型套件前確認 pinned 模型檔可從官方 Hub/CDN 取得。
# 若 Hugging Face 發生全球 outage，提早停止可避免浪費 A100 時間。
import time, urllib.request
HUB_PROBE_URL = (
    "https://huggingface.co/steven0226/qwen3vl-8b-chartqa-awq/resolve/"
    "43b71926a1d645133560347787539729bcd3de6b/config.json"
)
hub_error = None
for attempt in range(1, 4):
    try:
        request = urllib.request.Request(HUB_PROBE_URL, headers={"User-Agent": "chartqa-colab-preflight/1.0"})
        with urllib.request.urlopen(request, timeout=30) as response:
            assert response.status == 200, response.status
        hub_error = None
        print("Hugging Face Hub/CDN preflight: PASS")
        break
    except Exception as exc:
        hub_error = exc
        print(f"Hub/CDN preflight {attempt}/3 failed: {type(exc).__name__}")
        if attempt < 3:
            time.sleep(10 * attempt)
if hub_error is not None:
    raise RuntimeError(
        "Hugging Face Hub/CDN 目前無法取得模型；這不是 CUDA、Token 或 notebook 錯誤。"
        "請查看 https://status.huggingface.co/，待服務恢復後再重新全部執行。"
    ) from hub_error


In [ ]:
%%capture
# 2. 安裝已由正式評估／benchmark 驗證的 CUDA 12.9 serving 與 UI 相依
import os, subprocess, sys

os.environ.update({
    "HF_HUB_ENABLE_HF_TRANSFER": "0",
    "HF_HUB_DISABLE_XET": "1",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "VLLM_LOGGING_LEVEL": "WARNING",
    "VLLM_NO_USAGE_STATS": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "OMP_NUM_THREADS": "1",
})
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], check=True)
subprocess.run([
    "uv", "pip", "install", "--system", "--no-cache",
    "vllm==0.25.1+cu129",
    "torch==2.11.0+cu129", "torchvision==0.26.0+cu129", "torchaudio==2.11.0+cu129",
    "transformers==5.10.1", "gradio==6.20.0",
    "pillow==11.3.0", "httpx==0.28.1",
    "--extra-index-url", "https://wheels.vllm.ai/0.25.1/cu129",
    "--extra-index-url", "https://download.pytorch.org/whl/cu129",
    "--index-strategy", "unsafe-best-match",
], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "hf_xet"], check=False)


In [ ]:
# 3. 套件 preflight、HF_TOKEN 直傳與 pinned 模型設定（不呼叫 whoami）
import importlib.metadata, subprocess, sys
from google.colab import userdata

expected_versions = {
    "vllm": "0.25.1+cu129", "torch": "2.11.0+cu129",
    "torchvision": "0.26.0+cu129", "torchaudio": "2.11.0+cu129",
    "transformers": "5.10.1", "gradio": "6.20.0",
    "pillow": "11.3.0", "httpx": "0.28.1",
}
resolved_versions = {name: importlib.metadata.version(name) for name in expected_versions}
assert resolved_versions == expected_versions, resolved_versions
probe_code = (
    "import torch, torchvision; from vllm import LLM, SamplingParams; "
    "assert torch.version.cuda == '12.9', torch.version.cuda; "
    "assert torch.cuda.is_available(), 'CUDA unavailable'; "
    "print(torch.__version__, torch.version.cuda)"
)
probe = subprocess.run([sys.executable, "-c", probe_code], capture_output=True, text=True)
print("CUDA preflight:", probe.stdout.strip())
if probe.returncode != 0:
    print(probe.stderr)
assert probe.returncode == 0, "vLLM CUDA 12.9 import 失敗；不要繼續載入模型。"

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "找不到 Colab Secret: HF_TOKEN；請在左側 Secrets 建立並允許 notebook 存取。"
MODEL_REPO = "steven0226/qwen3vl-8b-chartqa-awq"
MODEL_REVISION = "43b71926a1d645133560347787539729bcd3de6b"
print("versions:", resolved_versions)
print(f"HF_TOKEN: found | model: https://huggingface.co/{MODEL_REPO} | revision: {MODEL_REVISION[:12]}")


In [ ]:
# 4. 預抓 pinned 模型；每分鐘心跳，避免下載時看起來像卡住
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeout
from pathlib import Path
from huggingface_hub import snapshot_download

def prefetch_repo(repo_id, revision, max_retries=8):
    last = None
    for attempt in range(1, max_retries + 1):
        try:
            started = time.time()
            with ThreadPoolExecutor(max_workers=1) as pool:
                future = pool.submit(
                    snapshot_download, repo_id, revision=revision, token=HF_TOKEN,
                )
                while True:
                    try:
                        path = Path(future.result(timeout=60)).resolve()
                        assert path.name == revision, f"snapshot revision 不符: {path.name} != {revision}"
                        print(f"model cached and revision verified: {path}")
                        return str(path)
                    except FutureTimeout:
                        print(f"model snapshot still downloading: {(time.time()-started)/60:.0f} min")
        except Exception as e:
            last = e
            wait = min(15 * attempt, 90)
            detail = str(e).replace("\n", " ")[:180]
            print(f"download {attempt}/{max_retries}: {type(e).__name__}: {detail}; {wait}s 後重試")
            time.sleep(wait)
    raise last

MODEL_PATH = prefetch_repo(MODEL_REPO, MODEL_REVISION)


In [ ]:
# 5. 在獨立 OS 子程序啟動 vLLM OpenAI server，避免 ipykernel stdout/fileno 衝突
import base64, io, json, os, shutil, signal, socket, subprocess, time
import urllib.error, urllib.request
from pathlib import Path
from PIL import Image, ImageDraw

SERVED_MODEL_NAME = "chartqa-awq"
HOST, PORT = "127.0.0.1", 8000
SERVER_START_TIMEOUT = 20 * 60
SERVER_LOG_PATH = Path("/content/vllm_demo_server.log")
VLLM_BIN = shutil.which("vllm")
assert VLLM_BIN, "找不到 vllm CLI；請確認安裝 cell 已成功。"
HTTP = urllib.request.build_opener(urllib.request.ProxyHandler({}))

def tail(path, lines=100):
    text = Path(path).read_text(encoding="utf-8", errors="replace") if Path(path).exists() else ""
    return "\n".join(text.splitlines()[-lines:])

def health_reachable():
    try:
        with HTTP.open(f"http://{HOST}:{PORT}/health", timeout=2) as response:
            return response.status == 200
    except Exception:
        return False

def port_bindable():
    probe = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        probe.bind((HOST, PORT))
        return True
    except OSError:
        return False
    finally:
        probe.close()

def descendants(root_pid):
    result = subprocess.run(["ps", "-eo", "pid=,ppid="], capture_output=True, text=True)
    children = {}
    for line in result.stdout.splitlines():
        try:
            pid, ppid = map(int, line.split())
        except ValueError:
            continue
        children.setdefault(ppid, []).append(pid)
    found, stack = set(), [root_pid]
    while stack:
        parent = stack.pop()
        for child in children.get(parent, []):
            if child not in found:
                found.add(child)
                stack.append(child)
    return found

def terminate_group(proc):
    if proc is None:
        return
    known = {proc.pid} | descendants(proc.pid)
    try:
        os.killpg(proc.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        pass
    known |= descendants(proc.pid)
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    for pid in known:
        try:
            os.kill(pid, signal.SIGKILL)
        except (ProcessLookupError, PermissionError):
            pass

def wait_for_server(proc):
    started, last_notice = time.time(), -1
    while time.time() - started < SERVER_START_TIMEOUT:
        if proc.poll() is not None:
            print(tail(SERVER_LOG_PATH))
            raise RuntimeError(f"vLLM server 提前退出（exit={proc.returncode}）")
        try:
            with HTTP.open(f"http://{HOST}:{PORT}/health", timeout=5) as response:
                if response.status == 200:
                    with HTTP.open(f"http://{HOST}:{PORT}/v1/models", timeout=10) as models_response:
                        models = json.loads(models_response.read().decode())
                    ids = [item["id"] for item in models.get("data", [])]
                    assert SERVED_MODEL_NAME in ids, ids
                    return
        except Exception:
            pass
        elapsed_min = int((time.time() - started) // 60)
        if elapsed_min > last_notice:
            print(f"[server] 載入中: {elapsed_min} min")
            last_notice = elapsed_min
        time.sleep(5)
    print(tail(SERVER_LOG_PATH))
    raise TimeoutError("vLLM server 20 分鐘內未就緒")

def post_chat(messages, max_tokens, timeout=300):
    payload = {
        "model": SERVED_MODEL_NAME, "messages": messages,
        "temperature": 0, "max_tokens": int(max_tokens),
    }
    request = urllib.request.Request(
        f"http://{HOST}:{PORT}/v1/chat/completions",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    try:
        with HTTP.open(request, timeout=timeout) as response:
            body = json.loads(response.read().decode())
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode(errors="replace")[:1000]
        raise RuntimeError(f"vLLM HTTP {exc.code}: {detail}") from exc
    assert body.get("choices"), body
    return body["choices"][0]["message"]["content"].strip()

assert not health_reachable(), "Port 8000 已有服務；請重新啟動 Colab runtime。"
assert port_bindable(), "Port 8000 無法綁定；請重新啟動 Colab runtime。"
gpu_processes = subprocess.run(
    ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
assert not gpu_processes, f"啟動前仍有 GPU process；請重新啟動 Colab runtime：\n{gpu_processes}"

SERVER_CMD = [
    VLLM_BIN, "serve", MODEL_PATH,
    "--host", HOST, "--port", str(PORT),
    "--served-model-name", SERVED_MODEL_NAME,
    "--dtype", "auto", "--max-model-len", "4096", "--max-num-seqs", "1",
    "--gpu-memory-utilization", "0.88",
    "--limit-mm-per-prompt", '{"image":1,"video":0}',
    "--mm-processor-cache-gb", "0", "--no-enable-prefix-caching",
    "--generation-config", "vllm", "--seed", "3407",
    "--disable-log-stats", "--disable-uvicorn-access-log",
    "--uvicorn-log-level", "warning",
]
server_env = os.environ.copy()
server_env.update({
    "PYTHONUNBUFFERED": "1", "VLLM_LOGGING_LEVEL": "INFO",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn", "VLLM_NO_USAGE_STATS": "1",
})
with open(SERVER_LOG_PATH, "w", encoding="utf-8") as log_file:
    VLLM_SERVER = subprocess.Popen(
        SERVER_CMD, stdout=log_file, stderr=subprocess.STDOUT,
        env=server_env, start_new_session=True,
    )
try:
    wait_for_server(VLLM_SERVER)
    probe_image = Image.new("RGB", (192, 128), "white")
    draw = ImageDraw.Draw(probe_image)
    draw.rectangle((28, 60, 62, 110), fill="#ff5a36")
    draw.rectangle((82, 35, 116, 110), fill="#263345")
    draw.rectangle((136, 18, 170, 110), fill="#b9f23f")
    probe_buffer = io.BytesIO()
    probe_image.save(probe_buffer, format="JPEG")
    probe_uri = "data:image/jpeg;base64," + base64.b64encode(probe_buffer.getvalue()).decode()
    probe_answer = post_chat([{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": probe_uri}},
        {"type": "text", "text": "How many bars are shown? Answer with one number."},
    ]}], max_tokens=4)
    assert probe_answer, "multimodal probe 沒有回答"
except BaseException:
    terminate_group(VLLM_SERVER)
    raise
print(f"server ready; multimodal probe PASS | {MODEL_REPO}@{MODEL_REVISION[:12]}")


In [ ]:
# 6. 推論函式與 CHART/OPS 介面
import base64, io, time
from PIL import Image
import gradio as gr

ANSWER_INSTRUCTION = "Answer the question using a single word or phrase."
SHORT = "ChartQA 短答"
EXPLAIN = "分析說明"

def to_data_uri(image, max_side=1280):
    image = image.convert("RGB")
    image.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    image.save(buf, format="JPEG", quality=92)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

def answer(image, question, response_mode, max_tokens):
    if image is None:
        raise gr.Error("請先上傳一張圖表。")
    if not (question or "").strip():
        raise gr.Error("請輸入你想問圖表的問題。")
    prompt = question.strip()
    if response_mode == SHORT:
        prompt += f"\n{ANSWER_INSTRUCTION}"
    else:
        prompt += ("\nRead the exact values from the chart. Explain the evidence and calculation briefly. "
                   "End with 'Final answer: ...'.")
    messages = [{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": to_data_uri(image)}},
        {"type": "text", "text": prompt},
    ]}]
    effective_max_tokens = int(max_tokens) if response_mode == SHORT else max(int(max_tokens), 128)
    started = time.perf_counter()
    try:
        result = post_chat(messages, max_tokens=effective_max_tokens)
    except Exception as exc:
        raise gr.Error(f"推論失敗：{type(exc).__name__}: {str(exc)[:500]}") from exc
    elapsed = time.perf_counter() - started
    return result, (f"{elapsed:.2f}s · deterministic · max_tokens={effective_max_tokens} · "
                    f"vLLM server · {MODEL_REPO}@{MODEL_REVISION[:12]}")

CSS = r"""
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500&family=Noto+Sans+TC:wght@400;500;600;700;800;900&display=swap');
:root { --ink:#101722; --paper:#f3f0e8; --signal:#ff5a36; --acid:#b9f23f; --line:#263345; --ui:'Noto Sans TC','Microsoft JhengHei',sans-serif; --mono:'IBM Plex Mono','Noto Sans TC',monospace; }
::selection { background:var(--acid); color:var(--ink); }
html, body { background:var(--paper) !important; }
.gradio-container { width:calc(100% - 32px) !important; max-width:1240px !important; margin:0 auto !important; background:var(--paper) !important; color:var(--ink) !important; font-family:var(--ui) !important; --body-background-fill:var(--paper) !important; --background-fill-primary:#fffdf7 !important; --background-fill-secondary:#f3f0e8 !important; --block-background-fill:#fffdf7 !important; --block-border-color:#263345 !important; --input-background-fill:#ffffff !important; --body-text-color:#101722 !important; --block-label-text-color:#475467 !important; --input-placeholder-color:#98a2b3 !important; --button-secondary-background-fill:#fffdf7 !important; --button-secondary-text-color:#101722 !important; --button-secondary-border-color:#263345 !important; }
.gradio-container:before { content:''; position:fixed; inset:0; pointer-events:none; opacity:.28; background-image:radial-gradient(#192231 0.55px, transparent 0.55px); background-size:8px 8px; }
.chartops-shell { border-top:9px solid var(--ink); padding:34px 2px 20px; position:relative; }
.eyebrow { font:500 12px var(--mono); letter-spacing:.13em; display:flex; justify-content:space-between; gap:20px; border-bottom:1px solid var(--line); padding-bottom:12px; }
.hero-title { color:var(--ink) !important; font-family:var(--ui); font-size:clamp(46px,7vw,86px); line-height:.98; letter-spacing:-.06em; margin:30px 0 20px; max-width:920px; font-weight:900; }
.hero-title em { color:var(--signal); font-style:normal; }
.deck { max-width:760px; font-size:17px; line-height:1.8; color:#475467; letter-spacing:.015em; }
.metric-strip { display:grid; grid-template-columns:repeat(4,1fr); border:1px solid var(--line); margin:28px 0 10px; background:#fffdf7; }
.metric { padding:14px 16px; border-right:1px solid var(--line); } .metric:last-child{border:0}
.metric b { display:block; font:500 11px var(--mono); letter-spacing:.08em; color:#667085; }
.metric span { display:block; margin-top:6px; color:var(--ink) !important; font-size:18px; font-weight:700; }
.workbench { margin-top:18px; gap:18px !important; }
.panel, .workbench > .column { border:1px solid var(--line) !important; border-radius:0 !important; background:#fffdf7 !important; box-shadow:7px 7px 0 var(--ink) !important; padding:18px !important; }
.panel-title { font:500 12px var(--mono); letter-spacing:.1em; margin-bottom:10px; color:#344054; }
.ask-btn { min-height:54px; border-radius:0 !important; border:1px solid var(--ink) !important; background:var(--signal) !important; color:#fff !important; font-family:var(--ui) !important; font-weight:800 !important; letter-spacing:.08em !important; box-shadow:4px 4px 0 var(--ink) !important; transition:transform .15s,box-shadow .15s !important; }
.ask-btn:hover { transform:translate(2px,2px); box-shadow:2px 2px 0 var(--ink) !important; }
.answer-box textarea { font-size:25px !important; line-height:1.35 !important; font-weight:650 !important; background:#101722 !important; color:#f8f5ec !important; border-radius:0 !important; min-height:220px !important; }
.runtime-note textarea { font:400 11px var(--mono) !important; color:#667085 !important; }
.chartops-shell > * { animation:rise-in .55s cubic-bezier(.2,.8,.2,1) both; }
.chartops-shell > *:nth-child(2){animation-delay:.06s}.chartops-shell > *:nth-child(3){animation-delay:.12s}.chartops-shell > *:nth-child(4){animation-delay:.18s}
@keyframes rise-in { from{opacity:0;transform:translateY(12px)} to{opacity:1;transform:translateY(0)} }
footer { display:none !important; }
@media(max-width:760px){ .gradio-container{width:calc(100% - 20px) !important}.eyebrow{align-items:flex-start;flex-direction:column;gap:7px}.metric-strip{grid-template-columns:1fr 1fr}.metric:nth-child(2){border-right:0}.metric:nth-child(-n+2){border-bottom:1px solid var(--line)}.hero-title{font-size:40px;letter-spacing:-.035em}.deck{font-size:15px}.panel,.workbench > .column{box-shadow:4px 4px 0 var(--ink) !important} }
@media(prefers-reduced-motion:reduce){ .chartops-shell > *{animation:none} }
"""

theme = gr.themes.Base(
    primary_hue=gr.themes.colors.orange,
    neutral_hue=gr.themes.colors.slate,
    radius_size=gr.themes.sizes.radius_none,
)

with gr.Blocks(title="CHART/OPS｜Qwen3-VL 圖表分析") as demo:
    gr.HTML("""<section class='chartops-shell' lang='zh-TW'>
      <div class='eyebrow'><span>CHART/OPS · QWEN3-VL 08B</span><span>圖表理解工作台 / 2026</span></div>
      <h1 class='hero-title'>問圖表，<br><em>讓數據回答。</em></h1>
      <p class='deck'>上傳圖表，直接提問。模型以 ChartQA 進行 Fine-tuning、壓縮為 W4A16，並透過 A100 與 vLLM 提供推論服務。</p>
      <div class='metric-strip'>
        <div class='metric'><b>Backbone</b><span>Qwen3-VL 8B</span></div>
        <div class='metric'><b>Fine-tuning</b><span>QLoRA · 15K</span></div>
        <div class='metric'><b>ChartQA 準確率</b><span>85.52%</span></div>
        <div class='metric'><b>Serving</b><span>AWQ · vLLM</span></div>
      </div>
    </section>""")
    with gr.Row(elem_classes=["workbench"]):
        with gr.Column(scale=5, elem_classes=["panel"]):
            gr.HTML("<div class='panel-title'>01 / 圖表來源</div>")
            image_input = gr.Image(type="pil", label="上傳圖表", height=390)
        with gr.Column(scale=5, elem_classes=["panel"]):
            gr.HTML("<div class='panel-title'>02 / 輸入問題</div>")
            question = gr.Textbox(label="問題", placeholder="例如：哪個類別的數值最高？", lines=4)
            response_mode = gr.Radio([SHORT, EXPLAIN], value=SHORT, label="回答模式")
            with gr.Accordion("生成參數", open=False):
                max_tokens = gr.Slider(8, 256, value=64, step=8, label="Max output tokens")
            ask = gr.Button("開始分析圖表 →", variant="primary", elem_classes=["ask-btn"])
    with gr.Row(elem_classes=["workbench"]):
        with gr.Column(elem_classes=["panel"]):
            gr.HTML("<div class='panel-title'>03 / 模型回答</div>")
            output = gr.Textbox(label="回答", lines=7, elem_classes=["answer-box"])
            runtime = gr.Textbox(label="Runtime trace", elem_classes=["runtime-note"])
    ask.click(answer, [image_input, question, response_mode, max_tokens], [output, runtime])
    question.submit(answer, [image_input, question, response_mode, max_tokens], [output, runtime])

print("UI ready")


In [ ]:
# 7. 啟動公開暫時網址；停止本格或刪除 runtime 後網址即失效
try:
    demo.queue(default_concurrency_limit=1).launch(share=True, debug=True, theme=theme, css=CSS)
finally:
    terminate_group(VLLM_SERVER)
    print("Gradio 與 vLLM server 已停止，GPU 記憶體已釋放。")
